In [1]:
import os
import json

In [ ]:
# 경로 설정
folder_path = '../../../apple_data/labeling_data'

# 파일 목록 가져오기
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]

# 모든 json 파일 읽어서 리스트에 저장하기
all_data = []
for file_name in json_files:
   if len(file_name.split("_")) < 6:
      file_path = os.path.join(folder_path, file_name)
      with open(file_path, 'r', encoding='utf-8') as file:
         data = json.load(file)
         data['file_name'] = file_name.replace('.json', '.png')
         all_data.append(data)

# 데이터 확인
for item in all_data:
   print(f"Data: {item}")

In [ ]:
import pandas as pd
#  'camera_model', 'camera_software'
df_origin = pd.DataFrame(all_data)
df_origin = df_origin.drop(columns=[
   'group_no', 'no', 'img_no', 'catecode', 'copyright', 'format', 'gps_lng', 'gps_lat', 'white_balance',
   'f_stop', 'exposure_time', 'focal_length', 'full_aperture', 'identifier', 'iso'
])

df_origin = df_origin.astype({
   'cate1': 'string', 'cate2': 'string', 'cate3': 'string',      # 범주형 - 문자열
   'width': 'float64', 'height': 'float64', 'weight': 'float64', # 소수점 있는 숫자 (부동소수점)
   'repo': 'string', 'img_height': 'int64', 'img_width': 'int64',
   'date': 'string',                  # 날짜 (MySQL은 DATE 타입이지만, Pandas에서는 일단 문자열로 두는 경우 많음)
   'resolution': 'string',            # 해상도 - 문자열 ("1920x1080" 같은 경우)
   'bit': 'string',                   # 비트 깊이 - 문자열 ("8bit", "16bit" 등)
   'truncated': 'string',             # 잘림 여부
   'angle_direction': 'string',       # 각도 방향
   'verticality_angle': 'float64', 'horizontality_angle': 'float64',    # 수직각, 수평각 - 실수
   'bndbox': 'string',                # 바운딩 박스 정보 - JSON 형태일 경우 문자열로 저장 가능
   'file_name': 'string'              # 파일명 - 문자열
})

df_origin

In [14]:
from sqlalchemy import create_engine

# DB 연결 정보 설정
DB_USER = 'farmorai_admin'
DB_PASSWORD = 'farmorai12345'
DB_HOST = '192.168.0.4'  # 또는 특정 IP
DB_PORT = '3306'         # 포트 번호
DB_NAME = 'farmdb'

# SQLAlchemy 엔진 생성
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# if_exists 옵션
# - 'replace': 테이블 있으면 삭제 후 재생성
# - 'append': 기존 테이블에 데이터 추가
# - 'fail': 테이블 있으면 에러 발생
df_origin.to_sql(name='apple_label_dataset', con=engine, if_exists='replace', index=True)

1564